# IOAI — 2025 Summer National Classifier Clone (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
!git clone -q --filter=blob:none --no-checkout --depth 1 https://github.com/Hungarian-AI-Olympiad/HAIO-Hungarian-AI-Olympiad haio
!cd haio && git sparse-checkout set 2025/nyari-orszagos/feladatok/adatok/klasszifikalo-klon >/dev/null && git checkout -q
import shutil, glob
for f in glob.glob('haio/2025/nyari-orszagos/feladatok/adatok/klasszifikalo-klon/*.pt'): shutil.copy(f, '.')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Classifier Clone (Klasszifikáló Klón)

> **HAIO 2025 — Summer National Finals (ML)**

Replicate a pretrained **neural net** (`secret_model.pt`, 11 features → 6 classes) with a **non-neural model** (tree/forest/boosting). Score = **agreement-accuracy**: how often your model's predictions match the NN's predictions on the 320-row evaluation split. `data.pt` = `(X_train, X_test_small, y_train, y_test_small)`; `train_test_split.pt` = `(X_train, X_test, …)` — predict `X_test` (320). **Submit** `submission.csv` (`id,label`).

Baseline: **knowledge distillation** — fit a RandomForest to the NN's predictions on X_train, then predict X_test. (The rules require a *non-neural* clone; agreement with the NN is the metric.)

In [ ]:
import torch, numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
secret = torch.jit.load('secret_model.pt'); secret.eval()
X_train, X_test_small, y_train, y_test_small = torch.load('data.pt', weights_only=False)
_, X_test, _, _ = torch.load('train_test_split.pt', weights_only=False)
def nn_pred(X):
    with torch.no_grad(): return secret(X.float()).argmax(1).numpy()
X_train.shape, X_test.shape

## Distill: fit a RandomForest to the NN's labels on X_train, predict X_test

In [ ]:
Xtr = X_train.numpy(); Xte = X_test.numpy()
y_nn_train = nn_pred(X_train)                 # teacher (NN) labels on train
clf = RandomForestClassifier(n_estimators=400, random_state=42, n_jobs=-1).fit(Xtr, y_nn_train)
# self-check: agreement with NN on the small held-out (X_test_small)
agree = accuracy_score(nn_pred(X_test_small), clf.predict(X_test_small.numpy()))
print(f'self-check agreement (X_test_small): {agree:.4f}')

In [ ]:
pred = clf.predict(Xte)
pd.DataFrame({'id': range(len(pred)), 'label': pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(pred))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)